## 1. Identificación y Descripción de la Fuente de Datos

La fuente de datos para este pipeline es la API pública y gratuita de Frankfurter, proporcionada por el Banco Central Europeo (BCE). Esta API permite obtener tasas de cambio históricas para diversas divisas.

**Endpoint Utilizado:** `https://api.frankfurter.app/`

**Parámetros Específicos:**
- **Rango de Fechas:** `2026-01-01` hasta `2026-08-31`
- **Moneda Base:** `USD` (Dólar Estadounidense)
- **Monedas de Destino:** `EUR` (Euro), `GBP` (Libra Esterlina), `JPY` (Yen Japonés), `CAD` (Dólar Canadiense), `BRL` (Real Brasileño)

La API no requiere autenticación (token/API key) y devuelve los datos en formato JSON, lo que la hace ideal para demostraciones de pipelines de datos.

## 2. Extracción de Datos Crudos

En este paso, consumiremos el endpoint de la API de Frankfurter utilizando la librería `requests` de Python. La respuesta JSON cruda se guardará en un archivo local para asegurar la reproducibilidad y evitar solicitudes repetitivas a la API durante el desarrollo y las pruebas del pipeline.

In [21]:
import requests
import json
import os

# Definir el endpoint de la API y los parámetros
API_URL = "https://api.frankfurter.app/2026-01-01..2026-08-31"
PARAMS = {
    "from": "USD",
    "to": "EUR,GBP,JPY,CAD,BRL"
}

RAW_DATA_DIR = "data/raw"
RAW_DATA_PATH = os.path.join(RAW_DATA_DIR, "tasas_raw.json")

# Crear el directorio si no existe
os.makedirs(RAW_DATA_DIR, exist_ok=True)

print(f"Realizando solicitud a la API: {API_URL} con parámetros {PARAMS}")
response = requests.get(API_URL, params=PARAMS)
response.raise_for_status() # Lanza una excepción si la solicitud no fue exitosa (código de estado >= 400)

raw_data = response.json()

# Guardar la respuesta cruda en un archivo JSON
with open(RAW_DATA_PATH, 'w') as f:
    json.dump(raw_data, f, indent=4)

print(f"Datos crudos guardados en: {RAW_DATA_PATH}")

Realizando solicitud a la API: https://api.frankfurter.app/2026-01-01..2026-08-31 con parámetros {'from': 'USD', 'to': 'EUR,GBP,JPY,CAD,BRL'}
Datos crudos guardados en: data/raw/tasas_raw.json


## 3. Exploración Inicial de Datos

En esta etapa, cargaremos los datos crudos (`tasas_raw.json`) en un DataFrame de Pandas para realizar una exploración inicial. Esto nos permitirá entender la estructura de los datos, identificar los tipos de datos de cada columna y verificar la existencia de valores nulos antes de realizar transformaciones.

In [22]:
import pandas as pd
import json
import os

# Ruta al archivo de datos crudos
RAW_DATA_DIR = "data/raw"
RAW_DATA_PATH = os.path.join(RAW_DATA_DIR, "tasas_raw.json")

# Cargar los datos crudos desde el archivo JSON
with open(RAW_DATA_PATH, 'r') as f:
    raw_data = json.load(f)

# El JSON tiene una estructura con 'rates' anidados, vamos a normalizarlos
# La clave 'rates' contiene fechas y dentro de cada fecha, las tasas de cambio

# Convertir el diccionario de tasas en una lista de diccionarios
# Cada diccionario contendrá la fecha y las tasas para esa fecha

# Extraer la fecha base y los símbolos de moneda
base_currency = raw_data.get('base')

# Crear una lista de registros, uno por cada fecha
data_records = []
for date, rates in raw_data['rates'].items():
    record = {'date': date}
    record.update(rates)
    data_records.append(record)

# Convertir la lista de registros en un DataFrame de Pandas
df_raw = pd.DataFrame(data_records)

# Convertir la columna 'date' a tipo datetime
df_raw['date'] = pd.to_datetime(df_raw['date'])

# Mostrar dimensiones del DataFrame
print("\nDimensiones del DataFrame (filas, columnas):")
print(df_raw.shape)

# Mostrar el esquema de tipos de datos
print("\nEsquema de tipos de datos:")
print(df_raw.dtypes)

# Mostrar los primeros 5 registros
print("\nPrimeros 5 registros del DataFrame:")
print(df_raw.head())

# Conteo de valores nulos por columna
print("\nConteo de valores nulos por columna:")
print(df_raw.isnull().sum())


Dimensiones del DataFrame (filas, columnas):
(170, 6)

Esquema de tipos de datos:
date    datetime64[ns]
BRL            float64
CAD            float64
EUR            float64
GBP            float64
JPY            float64
dtype: object

Primeros 5 registros del DataFrame:
        date     BRL     CAD      EUR      GBP     JPY
0 2025-12-31  5.4778  1.3692  0.85106  0.74264  156.67
1 2026-01-02  5.4384  1.3733  0.85317  0.74388  156.93
2 2026-01-05  5.4415  1.3792  0.85734  0.74383  156.83
3 2026-01-06  5.3986  1.3777  0.85419  0.73998  156.44
4 2026-01-07  5.3916  1.3809  0.85587  0.74153  156.55

Conteo de valores nulos por columna:
date    0
BRL     0
CAD     0
EUR     0
GBP     0
JPY     0
dtype: int64


## 4. Transformaciones Significativas

En esta sección, realizaremos las transformaciones clave para preparar los datos para el análisis. Estas incluyen la extracción de características temporales, el cálculo de métricas derivadas y la agregación de datos a nivel mensual.

In [23]:
# Crear una copia del DataFrame para las transformaciones
df_processed = df_raw.copy()

# Establecer la columna 'date' como índice si aùn no lo está
# Esto facilita las operaciones de series de tiempo
if not isinstance(df_processed.index, pd.DatetimeIndex):
    df_processed = df_processed.set_index('date').sort_index()

print("DataFrame después de establecer 'date' como índice:")
print(df_processed.head())
print("\nTipo de índice:", type(df_processed.index))

DataFrame después de establecer 'date' como índice:
               BRL     CAD      EUR      GBP     JPY
date                                                
2025-12-31  5.4778  1.3692  0.85106  0.74264  156.67
2026-01-02  5.4384  1.3733  0.85317  0.74388  156.93
2026-01-05  5.4415  1.3792  0.85734  0.74383  156.83
2026-01-06  5.3986  1.3777  0.85419  0.73998  156.44
2026-01-07  5.3916  1.3809  0.85587  0.74153  156.55

Tipo de índice: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>


In [24]:
# 4.1. Extraer granularidades temporales (año, mes, nombre del día)
df_processed['year'] = df_processed.index.year
df_processed['month'] = df_processed.index.month
df_processed['day_name'] = df_processed.index.day_name()

print("\nDataFrame con granularidades temporales añadidas:")
print(df_processed[['year', 'month', 'day_name']].head())


DataFrame con granularidades temporales añadidas:
            year  month   day_name
date                              
2025-12-31  2025     12  Wednesday
2026-01-02  2026      1     Friday
2026-01-05  2026      1     Monday
2026-01-06  2026      1    Tuesday
2026-01-07  2026      1  Wednesday


In [25]:
# 4.2. Calcular métricas derivadas (variación diaria porcentual de EUR frente a USD)
# La API ya proporciona EUR en USD (1 USD = X EUR), por lo que un aumento en EUR significa que el USD se ha devaluado contra el EUR.
# Para la variación porcentual, usaremos el método `.pct_change()`
# Multiplicamos por 100 para obtener el porcentaje
df_processed['EUR_daily_pct_change'] = df_processed['EUR'].pct_change() * 100

print("\nVariación diaria porcentual de EUR frente a USD (primeros 5 registros, después del primer NaN):")
print(df_processed[['EUR', 'EUR_daily_pct_change']].head())

# Eliminar el primer NaN resultante de pct_change si queremos solo valores calculados
df_processed.dropna(subset=['EUR_daily_pct_change'], inplace=True)
print("\nDataFrame después de eliminar NaN de 'EUR_daily_pct_change':")
print(df_processed[['EUR', 'EUR_daily_pct_change']].head())


Variación diaria porcentual de EUR frente a USD (primeros 5 registros, después del primer NaN):
                EUR  EUR_daily_pct_change
date                                     
2025-12-31  0.85106                   NaN
2026-01-02  0.85317              0.247926
2026-01-05  0.85734              0.488765
2026-01-06  0.85419             -0.367415
2026-01-07  0.85587              0.196678

DataFrame después de eliminar NaN de 'EUR_daily_pct_change':
                EUR  EUR_daily_pct_change
date                                     
2026-01-02  0.85317              0.247926
2026-01-05  0.85734              0.488765
2026-01-06  0.85419             -0.367415
2026-01-07  0.85587              0.196678
2026-01-08  0.85653              0.077115


In [26]:
# 4.3. Crear un DataFrame agregado con estadísticas mensuales (promedio, mínimo y máximo de cada divisa)
# Agrupar por año y mes para calcular las estadísticas mensuales
monthly_stats = df_processed.groupby(['year', 'month']).agg(
    EUR_avg=('EUR', 'mean'),
    EUR_min=('EUR', 'min'),
    EUR_max=('EUR', 'max'),
    GBP_avg=('GBP', 'mean'),
    GBP_min=('GBP', 'min'),
    GBP_max=('GBP', 'max'),
    JPY_avg=('JPY', 'mean'),
    JPY_min=('JPY', 'min'),
    JPY_max=('JPY', 'max'),
    CAD_avg=('CAD', 'mean'),
    CAD_min=('CAD', 'min'),
    CAD_max=('CAD', 'max'),
    BRL_avg=('BRL', 'mean'),
    BRL_min=('BRL', 'min'),
    BRL_max=('BRL', 'max')
).reset_index()

print("\nEstadísticas mensuales agregadas (primeros 5 registros):")
print(monthly_stats.head())

# Mostrar el DataFrame procesado final hasta ahora
print("\nDataFrame procesado final (primeros 5 registros con todas las columnas):")
print(df_processed.head())


Estadísticas mensuales agregadas (primeros 5 registros):
   year  month   EUR_avg  EUR_min  EUR_max   GBP_avg  GBP_min  GBP_max  \
0  2026      1  0.851994  0.83514  0.86081  0.739770  0.72376  0.74632   
1  2026      2  0.845753  0.84034  0.85085  0.736072  0.72893  0.74347   
2  2026      3  0.865201  0.85485  0.87138  0.749525  0.74345  0.75586   
3  2026      4  0.854266  0.84767  0.86768  0.742645  0.73691  0.75708   
4  2026      5  0.856686  0.84962  0.86244  0.741590  0.73415  0.74862   

      JPY_avg  JPY_min  JPY_max   CAD_avg  CAD_min  CAD_max   BRL_avg  \
0  156.717619   152.63   158.85  1.377895   1.3524   1.3904  5.339929   
1  155.155000   153.10   157.09  1.365080   1.3561   1.3711  5.203720   
2  158.675455   157.11   159.90  1.371164   1.3553   1.3935  5.234232   
3  159.063500   156.56   159.84  1.375060   1.3600   1.3912  5.025815   
4  158.239000   156.21   159.46  1.373030   1.3602   1.3854  4.985230   

   BRL_min  BRL_max  
0   5.1736   5.4415  
1   5.1258   5

## 5. Validaciones de Calidad de Datos

En esta etapa, implementaremos aserciones (`assert`) para garantizar la calidad de los datos procesados. Esto es crucial para asegurar la fiabilidad del pipeline y de cualquier análisis posterior.

In [27]:
# 5.1. Unicidad de fechas (clave temporal ùnica)
# El índice del DataFrame `df_processed` debe ser ùnico y de tipo datetime.

print("Realizando validación de unicidad de fechas...")
assert df_processed.index.is_unique, "Error de Calidad de Datos: Las fechas en el índice no son ùnicas."
assert isinstance(df_processed.index, pd.DatetimeIndex), "Error de Calidad de Datos: El índice no es de tipo DatetimeIndex."
print("Validación de unicidad y tipo de fechas: PASADA.")

Realizando validación de unicidad de fechas...
Validación de unicidad y tipo de fechas: PASADA.


In [28]:
# 5.2. Rango físico válido (todas las tasas deben ser mayores a cero: > 0)
# Seleccionamos solo las columnas de divisas para esta verificación.
currency_columns = ['BRL', 'CAD', 'EUR', 'GBP', 'JPY']

print("\nRealizando validación de rango físico (tasas > 0)...")
for col in currency_columns:
    assert (df_processed[col] > 0).all(), f"Error de Calidad de Datos: La columna '{col}' contiene valores no positivos."
print("Validación de rango físico (todas las tasas > 0): PASADA.")


Realizando validación de rango físico (tasas > 0)...
Validación de rango físico (todas las tasas > 0): PASADA.


In [29]:
# 5.3. Cero valores nulos en el dataset final procesado

print("\nRealizando validación de valores nulos en el DataFrame final...")
null_counts = df_processed.isnull().sum()
assert null_counts.sum() == 0, f"Error de Calidad de Datos: Se encontraron valores nulos en el DataFrame procesado.\n{null_counts[null_counts > 0]}"
print("Validación de cero valores nulos: PASADA.")

print("\nTodas las validaciones de calidad de datos han sido PASADAS con éxito.")


Realizando validación de valores nulos en el DataFrame final...
Validación de cero valores nulos: PASADA.

Todas las validaciones de calidad de datos han sido PASADAS con éxito.


## 6. Almacenamiento Procesado

Una vez que los datos han sido extraídos, transformados y validados, el siguiente paso es almacenarlos en un formato optimizado para el análisis posterior. Elegiremos Apache Parquet por sus ventajas en compresión, rendimiento y conservación del esquema de datos.

In [30]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

# Directorio para los datos procesados
PROCESSED_DATA_DIR = "data/processed"
PROCESSED_DATA_PATH = os.path.join(PROCESSED_DATA_DIR, "tasas_procesadas.parquet")

# Crear el directorio si no existe
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

# Convertir el DataFrame de Pandas a una tabla Apache Arrow
table = pa.Table.from_pandas(df_processed)

# Guardar la tabla en formato Parquet
pq.write_table(table, PROCESSED_DATA_PATH)

print(f"Dataset procesado guardado en: {PROCESSED_DATA_PATH}")
print(f"Dimensiones del archivo Parquet (estimado, basado en df_processed): {df_processed.shape}")

# Verificación de la lectura exitosa del archivo Parquet
print("\nVerificando la lectura del archivo Parquet guardado...")
df_verified = pd.read_parquet(PROCESSED_DATA_PATH)
print("Lectura exitosa del archivo Parquet. Primeros 5 registros:")
print(df_verified.head())
print(f"Dimensiones del DataFrame leído: {df_verified.shape}")

Dataset procesado guardado en: data/processed/tasas_procesadas.parquet
Dimensiones del archivo Parquet (estimado, basado en df_processed): (169, 9)

Verificando la lectura del archivo Parquet guardado...
Lectura exitosa del archivo Parquet. Primeros 5 registros:
               BRL     CAD      EUR      GBP     JPY  year  month   day_name  \
date                                                                           
2026-01-02  5.4384  1.3733  0.85317  0.74388  156.93  2026      1     Friday   
2026-01-05  5.4415  1.3792  0.85734  0.74383  156.83  2026      1     Monday   
2026-01-06  5.3986  1.3777  0.85419  0.73998  156.44  2026      1    Tuesday   
2026-01-07  5.3916  1.3809  0.85587  0.74153  156.55  2026      1  Wednesday   
2026-01-08  5.3800  1.3861  0.85653  0.74407  156.72  2026      1   Thursday   

            EUR_daily_pct_change  
date                              
2026-01-02              0.247926  
2026-01-05              0.488765  
2026-01-06             -0.367415  
2

## 7. Justificación Técnica: Parquet sobre CSV

La elección de **Apache Parquet** sobre CSV (Comma Separated Values) para almacenar el dataset analítico curado se basa en varias ventajas técnicas clave:

1.  **Formato Columnar:** Parquet es un formato de almacenamiento de datos columnar, a diferencia de CSV que es row-oriented. Esto significa que los datos se almacenan por columnas en lugar de por filas. Para cargas de trabajo analíticas (OLAP) donde a menudo se consultan solo un subconjunto de columnas, esto se traduce en:
    *   **Menos I/O:** Se lee solo la cantidad mínima de datos necesaria del disco.
    *   **Mayor rendimiento de consulta:** Motores de consulta como Spark, Presto o Dask pueden leer y procesar datos columnares de manera mucho más eficiente.

2.  **Compresión Superior:** Debido a su naturaleza columnar, Parquet puede aplicar algoritmos de compresión más efectivos y específicos por tipo de dato a cada columna. Esto resulta en tamaños de archivo significativamente menores que los CSV, lo que ahorra espacio de almacenamiento y acelera la transferencia de datos.

3.  **Conservación del Esquema y Tipos de Datos:** A diferencia de CSV, que no tiene un esquema auto-descriptivo y trata todo como cadenas de texto, Parquet almacena el esquema (nombres de columnas, tipos de datos) junto con los datos. Esto elimina la necesidad de inferencia de esquema en cada lectura, reduce errores de tipo de datos y garantiza la integridad del esquema a lo largo del pipeline.

4.  **Codificación Eficiente:** Parquet utiliza varias estrategias de codificación (por ejemplo, RLE, Bit-packing) para optimizar el almacenamiento de diferentes tipos de datos, lo que contribuye a la compresión y el rendimiento.

5.  **Soporte de Metadatos:** Permite almacenar metadatos adicionales, lo que puede ser ùtil para catalogación y gestión de datos.

En resumen, Parquet es un formato de archivo mucho más adecuado para ecosistemas de Big Data y análisis, ofreciendo eficiencia en almacenamiento, rendimiento en consultas y robustez en la gestión de esquemas, lo que lo hace ideal para el almacenamiento de datasets procesados y analíticos.

## 7. Justificación Técnica: Parquet sobre CSV

La elección de **Apache Parquet** sobre CSV (Comma Separated Values) para almacenar el dataset analítico curado se basa en varias ventajas técnicas clave:

1.  **Formato Columnar:** Parquet es un formato de almacenamiento de datos columnar, a diferencia de CSV que es row-oriented. Esto significa que los datos se almacenan por columnas en lugar de por filas. Para cargas de trabajo analíticas (OLAP) donde a menudo se consultan solo un subconjunto de columnas, esto se traduce en:
    *   **Menos I/O:** Se lee solo la cantidad mínima de datos necesaria del disco.
    *   **Mayor rendimiento de consulta:** Motores de consulta como Spark, Presto o Dask pueden leer y procesar datos columnares de manera mucho más eficiente.

2.  **Compresión Superior:** Debido a su naturaleza columnar, Parquet puede aplicar algoritmos de compresión más efectivos y específicos por tipo de dato a cada columna. Esto resulta en tamaños de archivo significativamente menores que los CSV, lo que ahorra espacio de almacenamiento y acelera la transferencia de datos.

3.  **Conservación del Esquema y Tipos de Datos:** A diferencia de CSV, que no tiene un esquema auto-descriptivo y trata todo como cadenas de texto, Parquet almacena el esquema (nombres de columnas, tipos de datos) junto con los datos. Esto elimina la necesidad de inferencia de esquema en cada lectura, reduce errores de tipo de datos y garantiza la integridad del esquema a lo largo del pipeline.

4.  **Codificación Eficiente:** Parquet utiliza varias estrategias de codificación (por ejemplo, RLE, Bit-packing) para optimizar el almacenamiento de diferentes tipos de datos, lo que contribuye a la compresión y el rendimiento.

5.  **Soporte de Metadatos:** Permite almacenar metadatos adicionales, lo que puede ser útil para catalogación y gestión de datos.

En resumen, Parquet es un formato de archivo mucho más adecuado para ecosistemas de Big Data y análisis, ofreciendo eficiencia en almacenamiento, rendimiento en consultas y robustez en la gestión de esquemas, lo que lo hace ideal para el almacenamiento de datasets procesados y analíticos.

## 7. Justificación Técnica: Parquet sobre CSV

La elección de **Apache Parquet** sobre CSV (Comma Separated Values) para almacenar el dataset analítico curado se basa en varias ventajas técnicas clave:

1.  **Formato Columnar:** Parquet es un formato de almacenamiento de datos columnar, a diferencia de CSV que es row-oriented. Esto significa que los datos se almacenan por columnas en lugar de por filas. Para cargas de trabajo analíticas (OLAP) donde a menudo se consultan solo un subconjunto de columnas, esto se traduce en:
    *   **Menos I/O:** Se lee solo la cantidad mínima de datos necesaria del disco.
    *   **Mayor rendimiento de consulta:** Motores de consulta como Spark, Presto o Dask pueden leer y procesar datos columnares de manera mucho más eficiente.

2.  **Compresión Superior:** Debido a su naturaleza columnar, Parquet puede aplicar algoritmos de compresión más efectivos y específicos por tipo de dato a cada columna. Esto resulta en tamaños de archivo significativamente menores que los CSV, lo que ahorra espacio de almacenamiento y acelera la transferencia de datos.

3.  **Conservación del Esquema y Tipos de Datos:** A diferencia de CSV, que no tiene un esquema auto-descriptivo y trata todo como cadenas de texto, Parquet almacena el esquema (nombres de columnas, tipos de datos) junto con los datos. Esto elimina la necesidad de inferencia de esquema en cada lectura, reduce errores de tipo de datos y garantiza la integridad del esquema a lo largo del pipeline.

4.  **Codificación Eficiente:** Parquet utiliza varias estrategias de codificación (por ejemplo, RLE, Bit-packing) para optimizar el almacenamiento de diferentes tipos de datos, lo que contribuye a la compresión y el rendimiento.

5.  **Soporte de Metadatos:** Permite almacenar metadatos adicionales, lo que puede ser útil para catalogación y gestión de datos.

En resumen, Parquet es un formato de archivo mucho más adecuado para ecosistemas de Big Data y análisis, ofreciendo eficiencia en almacenamiento, rendimiento en consultas y robustez en la gestión de esquemas, lo que lo hace ideal para el almacenamiento de datasets procesados y analíticos.